In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))   # repo root, for `kernels` / `model`

import math

import pandas as pd
import torch, triton
import torch.nn.functional as F
from torch.nn.attention import sdpa_kernel, SDPBackend
from triton.testing import do_bench

from kernels import fa2_fwd, _fa2_fwd

H, H_KV, D = 32, 8, 64
PEAK = 312e12
torch.manual_seed(0)
print(torch.cuda.get_device_name(), torch.cuda.get_device_capability(),
      torch.__version__, triton.__version__)
print(torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

def qkv(S, B=1):
    q = torch.randn(B, H,    S, D, device=DEV, dtype=torch.float16)
    k = torch.randn(B, H_KV, S, D, device=DEV, dtype=torch.float16)
    v = torch.randn(B, H_KV, S, D, device=DEV, dtype=torch.float16)
    return q, k, v

def useful_flops(S, B=1):
    return 4 * B * S * S * D * H / 2

NVIDIA A100-SXM4-40GB (8, 0) 2.8.0+cu128 3.4.0
42.406903808 GB


In [7]:
CONFIGS = [triton.Config({"BLOCK_M": BM, "BLOCK_N": BN}, num_warps=w, num_stages=s)
           for BM in (64, 128) for BN in (32, 64, 128)
           for w in (4, 8) for s in (2, 3)
           if BM % BN == 0]
print(len(CONFIGS), "configs")

_fa2_tuned = triton.autotune(configs=CONFIGS, key=["S_q", "S_kv"])(_fa2_fwd)

def fa2_fwd_tuned(q, k, v, causal=True):
    B, H_, S_q, d = q.shape
    _, H_kv, S_kv, _ = k.shape
    o = torch.empty_like(q)
    lse = torch.empty((B, H_, S_q), dtype=torch.float32, device=q.device)
    grid = lambda META: (triton.cdiv(S_q, META["BLOCK_M"]), B * H_)
    _fa2_tuned[grid](q, k, v, o, lse,
                     *q.stride(), *k.stride(), *v.stride(), *o.stride(),
                     S_q, S_kv, float(1.0 / math.sqrt(d)),
                     H=H_, N_REP=H_ // H_kv, CAUSAL=causal, HEAD_DIM=d)
    return o, lse

20 configs


In [8]:
def ref_chunked(q, k, v, causal=True, chunk=8):
    """float64 ground truth, a few heads at a time."""
    n_rep = q.shape[1] // k.shape[1]
    outs, lses = [], []
    for h0 in range(0, q.shape[1], chunk):
        h1 = h0 + chunk
        qc = q[:, h0:h1].double()
        kc = k[:, h0 // n_rep: h1 // n_rep].double().repeat_interleave(n_rep, dim=1)
        vc = v[:, h0 // n_rep: h1 // n_rep].double().repeat_interleave(n_rep, dim=1)
        s = qc @ kc.transpose(-2, -1) / math.sqrt(qc.shape[-1])
        if causal:
            S_ = s.shape[-1]
            m = torch.ones(S_, S_, dtype=torch.bool, device=s.device).triu(1)
            s = s.masked_fill(m, float("-inf"))
        lses.append(torch.logsumexp(s, -1))
        outs.append(torch.softmax(s, -1) @ vc)
        del s
    return torch.cat(outs, 1), torch.cat(lses, 1)

def rel(x, r):
    x, r = x.double(), r.double()
    return ((x - r).abs().max() / r.pow(2).mean().sqrt()).item()

def check(S, BM=64, BN=64, w=4, s=2, tuned=False):
    q, k, v = qkv(S)
    o_ref, l_ref = ref_chunked(q, k, v)
    kr = k.repeat_interleave(H // H_KV, dim=1).contiguous()
    vr = v.repeat_interleave(H // H_KV, dim=1).contiguous()
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        floor = rel(F.scaled_dot_product_attention(q, kr, vr, is_causal=True), o_ref)
    o, lse = (fa2_fwd_tuned(q, k, v, causal=True) if tuned else
              fa2_fwd(q, k, v, causal=True, BLOCK_M=BM, BLOCK_N=BN,
                      num_warps=w, num_stages=s))
    o_r, l_r = rel(o, o_ref), rel(lse, l_ref)
    ok = torch.isfinite(o).all().item() and o_r <= max(2 * floor, 1e-3) and l_r <= 1e-3
    print(f"{'PASS' if ok else 'FAIL'}  S={S:5d} {'tuned' if tuned else f'{BM}/{BN}/{w}/{s}'}"
          f"   o_rel={o_r:.2e}  lse_rel={l_r:.2e}  flash_floor={floor:.2e}")
    del q, k, v, kr, vr, o_ref, l_ref; torch.cuda.empty_cache()
    return ok

for S in (1000, 4096):
    check(S)
check(1000, BM=128, BN=64, w=8, s=2)
check(4096, tuned=True)

PASS  S= 1000 64/64/4/2   o_rel=7.38e-03  lse_rel=1.87e-07  flash_floor=7.38e-03
PASS  S= 4096 64/64/4/2   o_rel=1.52e-02  lse_rel=1.54e-07  flash_floor=1.52e-02
PASS  S= 1000 128/64/8/2   o_rel=8.39e-03  lse_rel=2.36e-07  flash_floor=8.39e-03
PASS  S= 4096 tuned   o_rel=1.87e-02  lse_rel=1.54e-07  flash_floor=1.87e-02


True

In [9]:
SEQS = [1024, 4096, 8192]
rows = []
for S in SEQS:
    q, k, v = qkv(S)
    for c in CONFIGS:
        BM, BN = c.kwargs["BLOCK_M"], c.kwargs["BLOCK_N"]
        w, s = c.num_warps, c.num_stages
        fn = lambda BM=BM, BN=BN, w=w, s=s: fa2_fwd(
            q, k, v, causal=True, BLOCK_M=BM, BLOCK_N=BN, num_warps=w, num_stages=s)
        try:
            fn(); torch.cuda.synchronize()
            p50, p20, p80 = do_bench(fn, quantiles=[0.5, 0.2, 0.8])
        except Exception as e:
            print(f"skip S={S} {BM}/{BN}/{w}/{s}: {type(e).__name__}")
            continue
        tfl = useful_flops(S) / (p50 * 1e-3) / 1e12
        rows.append(dict(S=S, BM=BM, BN=BN, warps=w, stages=s, us=p50 * 1e3,
                         p20=p20 * 1e3, p80=p80 * 1e3, tflops=tfl, pct_peak=100 * tfl * 1e12 / PEAK))
    del q, k, v; torch.cuda.empty_cache()

sweep = pd.DataFrame(rows).sort_values(["S", "us"])
sweep.to_csv("../results/sweep_configs.csv", index=False)
for S in SEQS:
    print(f"\n--- S={S} (top 5) ---")
    print(sweep[sweep.S == S].head(5).to_string(index=False))
best = {S: sweep[sweep.S == S].iloc[0] for S in SEQS}
print("\nwinners:", {S: (int(b.BM), int(b.BN), int(b.warps), int(b.stages)) for S, b in best.items()})


--- S=1024 (top 5) ---
   S  BM  BN  warps  stages        us       p20       p80    tflops  pct_peak
1024  64  64      4       2 69.632001 68.608001 70.656002 61.680940 19.769532
1024  64  64      4       3 72.704002 71.680002 74.752003 59.074702 18.934199
1024 128  64      8       3 74.752003 73.728003 74.752003 57.456217 18.415454
1024  64  32      4       3 75.776003 71.680002 92.160001 56.679781 18.166597
1024 128 128      8       3 79.871997 79.871997 80.895998 53.773130 17.234978

--- S=4096 (top 5) ---
   S  BM  BN  warps  stages         us        p20        p80     tflops  pct_peak
4096  64  64      4       2 628.736019 621.568024 636.313617 109.297821 35.031353
4096  64  32      4       2 648.703992 639.999986 662.527978 105.933488 33.953041
4096  64  32      4       3 684.032023 676.659226 694.271982 100.462368 32.199477
4096 128  64      8       3 685.055971 684.032023 686.079979 100.312208 32.151349
4096  64  64      4       3 708.607972 703.487992 712.704003  96.978131 31

In [11]:
def eager(q, kr, vr, mask):
    s = (q @ kr.transpose(-2, -1)) * (1.0 / math.sqrt(D)) + mask
    return torch.softmax(s, dim=-1) @ vr

rows = []
for S in SEQS:
    q, k, v = qkv(S)
    kr = k.repeat_interleave(H // H_KV, dim=1).contiguous()
    vr = v.repeat_interleave(H // H_KV, dim=1).contiguous()
    mask = torch.zeros(S, S, device=DEV, dtype=torch.float16).masked_fill(
        torch.ones(S, S, device=DEV, dtype=torch.bool).triu(1), float("-inf"))
    b = best[S]

    def flash(q=q, kr=kr, vr=vr):
        with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
            return F.scaled_dot_product_attention(q, kr, vr, is_causal=True)

    contestants = {
        "eager":      lambda q=q, kr=kr, vr=vr, m=mask: eager(q, kr, vr, m),
        "sdpa_flash": flash,
        "ours_64/64": lambda q=q, k=k, v=v: fa2_fwd(q, k, v, causal=True),
        "ours_best":  lambda q=q, k=k, v=v, b=b: fa2_fwd(
            q, k, v, causal=True, BLOCK_M=int(b.BM), BLOCK_N=int(b.BN),
            num_warps=int(b.warps), num_stages=int(b.stages)),
        "ours_tuned": lambda q=q, k=k, v=v: fa2_fwd_tuned(q, k, v, causal=True),
    }
    for name, fn in contestants.items():
        try:
            fn(); torch.cuda.synchronize()
            p50, p20, p80 = do_bench(fn, quantiles=[0.5, 0.2, 0.8])
        except Exception as e:
            print(f"skip S={S} {name}: {type(e).__name__}: {e}")
            torch.cuda.empty_cache(); continue
        tfl = useful_flops(S) / (p50 * 1e-3) / 1e12
        rows.append(dict(S=S, kernel=name, us=p50 * 1e3, p20=p20 * 1e3, p80=p80 * 1e3,
                         tflops=tfl, pct_peak=100 * tfl * 1e12 / PEAK))
    del q, k, v, kr, vr, mask; torch.cuda.empty_cache()

bench = pd.DataFrame(rows)
assert len(bench), "no contestant ran -- read the skip messages above"
flash_us = bench[bench.kernel == "sdpa_flash"].set_index("S").us
bench["vs_flash"] = bench.S.map(flash_us) / bench.us
bench.to_csv("../results/bench_attn.csv", index=False)
print(bench.to_string(index=False))

   S     kernel           us          p20          p80     tflops  pct_peak  vs_flash
1024      eager   623.615980   622.591972   624.639988   6.887199  2.207436  0.124795
1024 sdpa_flash    77.823997    75.776003    80.895998  55.188213 17.688530  1.000000
1024 ours_64/64    69.632001    67.584001    70.656002  61.680940 19.769532  1.117647
1024  ours_best    69.632001    68.608001    71.680002  61.680940 19.769532  1.117647
1024 ours_tuned    69.632001    68.608001    71.680002  61.680940 19.769532  1.117647
4096      eager  9561.087608  9559.859467  9570.508766   7.187412  2.303658  0.050230
4096 sdpa_flash   480.255991   472.883195   486.400008 143.089265 45.861944  1.000000
4096 ours_64/64   629.760027   623.615980   636.928022 109.120099 34.974391  0.762602
4096  ours_best   627.712011   621.568024   634.880006 109.476122 35.088501  0.765090
4096 ours_tuned   626.688004   620.544016   633.855999 109.655006 35.145835  0.766340
8192      eager 40494.590759 40491.825867 40497.355652

In [ ]:
import importlib
import kernels

importlib.reload(kernels)
print("import ok", kernels.fa2_fwd)

In [19]:
HBM = 1.555e12
def kv_requests(S, BM=64, BN=64, B=1, H=32, H_KV=8, d=64):
    nblk_m = S // BM
    blocks = sum((p * BM) // BN + (BM // BN) for p in range(nblk_m))
    per_head = blocks * BN * d * 2 * 2
    return B * H * per_head, B * H * S * d * 2

t = bench[bench.kernel == "ours_64/64"].set_index("S").us
for S in SEQS:
    kvq, qq = kv_requests(S)
    req = kvq + qq
    us = t[S]
    ideal = (2 * 1 * 32 * S * 64 * 2) + (2 * 1 * 8 * S * 64 * 2) + 1 * 32 * S * 4
    print(f"S={S:5d}  requested {req/1e9:6.3f} GB  in {us:8.1f} us "
          f"-> {req/(us*1e-6)/1e12:5.2f} TB/s = {100*req/(us*1e-6)/HBM:6.1f}% of HBM peak   "
          f"| ideal DRAM {ideal/1e6:5.1f} MB = {100*ideal/(us*1e-6)/HBM:4.1f}% of peak")

import itertools
res = []
for B, H_kv in itertools.product((1, 2, 4), (8, 32)):
    S = 4096
    q = torch.randn(B, 32, S, 64, device=DEV, dtype=torch.float16)
    k = torch.randn(B, H_kv, S, 64, device=DEV, dtype=torch.float16)
    v = torch.randn(B, H_kv, S, 64, device=DEV, dtype=torch.float16)
    fn = lambda q=q, k=k, v=v: fa2_fwd(q, k, v, causal=True)
    try:
        fn(); torch.cuda.synchronize()
        p50 = do_bench(fn, quantiles=[0.5, 0.2, 0.8])[0]
    except Exception as e:
        print("skip", B, H_kv, type(e).__name__); torch.cuda.empty_cache(); continue
    kv_mb = B * H_kv * S * 64 * 2 * 2 / 1e6
    tfl = useful_flops(S, B) / (p50 * 1e-3) / 1e12
    res.append(dict(B=B, H_kv=H_kv, kv_MB=kv_mb, us=p50 * 1e3, tflops=tfl,
                    fits_L2=kv_mb < 40))
    del q, k, v; torch.cuda.empty_cache()

r = pd.DataFrame(res).sort_values("kv_MB")
r.to_csv("../results/l2_residency.csv", index=False)
print(r.to_string(index=False))

S= 1024  requested  0.075 GB  in     69.6 us ->  1.08 TB/s =   69.7% of HBM peak   | ideal DRAM  10.6 MB =  9.8% of peak
S= 4096  requested  1.107 GB  in    629.8 us ->  1.76 TB/s =  113.1% of HBM peak   | ideal DRAM  42.5 MB =  4.3% of peak
S= 8192  requested  4.362 GB  in   2276.9 us ->  1.92 TB/s =  123.2% of HBM peak   | ideal DRAM  84.9 MB =  2.4% of peak
 B  H_kv      kv_MB          us     tflops  fits_L2
 1     8   8.388608  807.936013  85.055593     True
 2     8  16.777216 1288.192034 106.691355     True
 1    32  33.554432  696.319997  98.689506     True
 4     8  33.554432 2256.896019 121.794670     True
 2    32  67.108864 1171.455979 117.323191    False
 4    32 134.217728 2272.255898 120.971369    False
